In [9]:
import sys

# Install transformers if not already installed
!{sys.executable} -m pip install transformers torch gradio

### 2. Model Loading (DialoGPT-medium)

We will load a pre-trained `DialoGPT-medium` model and its corresponding tokenizer from Hugging Face. This model is specifically designed for conversational AI.

In [10]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

try:
    # Load pre-trained model and tokenizer
    model_name = 'microsoft/DialoGPT-medium'
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)
    print(f"Successfully loaded model: {model_name}")
except Exception as e:
    print(f"Error loading model or tokenizer: {e}")
    tokenizer = None
    model = None

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Successfully loaded model: microsoft/DialoGPT-medium


### 3. Implement the Chatbot Logic

This section defines the core chatbot functionality. It includes:
*   **User Input Handling**: Takes input from the console.
*   **Response Generation**: Uses the loaded transformer model to generate responses.
*   **Continuous Conversation**: Maintains `chat_history_ids` to provide context for subsequent responses.
*   **Exit Condition**: Stops the chatbot when the user types 'exit' or 'quit'.
*   **Error Handling**: Basic `try-except` blocks for generation.
*   **Multi-turn Context Management**: Limits `chat_history_ids` length to prevent it from growing indefinitely, by keeping only the last few turns (e.g., last 5 turns).

In [11]:
def run_console_chatbot(tokenizer, model):
    if not tokenizer or not model:
        print("Chatbot cannot start. Model or tokenizer failed to load.")
        return

    print("Hello! I am your AI assistant. How can I help you today?")
    chat_history_ids = None
    # Max history length to manage context and prevent memory issues
    max_history_length = 5 # Keep last 5 turns of conversation

    while True:
        try:
            user_input = input("User: ")
            if user_input.lower() in ['exit', 'quit']:
                print("Chatbot: Goodbye!")
                break

            # Encode the new user input, adding the eos_token and return_tensors='pt'
            new_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt')

            # Append the new user input to the chat history
            if chat_history_ids is None:
                bot_input_ids = new_input_ids
            else:
                # Only keep a certain number of previous turns
                # For DialoGPT, chat_history_ids already includes EOS tokens for turns.
                # A simple way to manage context length: truncate if it gets too long.
                # A more sophisticated approach would involve counting tokens and managing by token limit.
                # Here, we keep a fixed number of recent history IDs for simplicity.

                # Get the number of tokens in the current history
                current_history_len = chat_history_ids.shape[-1]
                # Get the number of tokens for the new input
                new_input_len = new_input_ids.shape[-1]

                # If total length exceeds a threshold, prune the history
                # For simplicity, let's keep it to a certain number of tokens, e.g., 512
                # DialoGPT's max context is 1024, but keeping it shorter helps with efficiency.
                max_sequence_length = 512

                if current_history_len + new_input_len > max_sequence_length:
                    # Keep a portion of the history that fits with the new input
                    # This is a basic truncation; more advanced methods might summarize or select relevant parts.
                    chat_history_ids = chat_history_ids[:, -(max_sequence_length - new_input_len):]

                bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)

            # Generate a response
            # The `pad_token_id` is important for DialoGPT
            chat_history_ids = model.generate(
                bot_input_ids,
                max_new_tokens=100, # Limit the length of the bot's response
                pad_token_id=tokenizer.eos_token_id,
                no_repeat_ngram_size=3,
                do_sample=True,
                top_k=50,
                top_p=0.95,
                temperature=0.7
            )

            # Decode the response
            response = tokenizer.decode(chat_history_ids[:, bot_input_ids.shape[-1]:][0], skip_special_tokens=True)
            print(f"Chatbot: {response}")

        except Exception as e:
            print(f"Chatbot encountered an error: {e}")
            print("Please try again or type 'exit' to quit.")

# Run the chatbot (this will block the notebook until 'exit' or 'quit' is typed)
# print("Starting console chatbot...")
# run_console_chatbot(tokenizer, model)

### 4. Test with the Given Scenario

Let's run the chatbot with the scenario provided in the assignment description to demonstrate its conversation flow and response generation. The `run_scenario` function simulates user input and captures chatbot responses.

In [12]:
from io import StringIO
import sys

def run_scenario(tokenizer, model, scenario_inputs):
    if not tokenizer or not model:
        print("Scenario cannot run. Model or tokenizer failed to load.")
        return

    # Capture stdout to get chatbot responses
    old_stdout = sys.stdout
    redirected_output = sys.stdout = StringIO()

    # Simulate user input
    input_queue = iter(scenario_inputs)
    def mock_input(prompt):
        try:
            print(prompt, end='') # Print the prompt as it would appear
            user_val = next(input_queue)
            print(user_val) # Print the user's input so it appears in logs
            return user_val
        except StopIteration:
            raise EOFError('No more input in scenario')

    # Replace input with our mock version
    original_input = __builtins__.input
    __builtins__.input = mock_input

    print("\n--- Running Scenario ---")
    chat_history_ids = None
    max_sequence_length = 512 # Same as in run_console_chatbot

    try:
        print("Hello! I am your AI assistant. How can I help you today?")
        while True:
            user_input = __builtins__.input("User: ") # Use the mock input

            if user_input.lower() in ['exit', 'quit']:
                print("Chatbot: Goodbye!")
                break

            new_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt')

            if chat_history_ids is None:
                bot_input_ids = new_input_ids
            else:
                current_history_len = chat_history_ids.shape[-1]
                new_input_len = new_input_ids.shape[-1]

                if current_history_len + new_input_len > max_sequence_length:
                    chat_history_ids = chat_history_ids[:, -(max_sequence_length - new_input_len):]
                bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)

            chat_history_ids = model.generate(
                bot_input_ids,
                max_new_tokens=100,
                pad_token_id=tokenizer.eos_token_id,
                no_repeat_ngram_size=3,
                do_sample=True,
                top_k=50,
                top_p=0.95,
                temperature=0.7
            )
            response = tokenizer.decode(chat_history_ids[:, bot_input_ids.shape[-1]:][0], skip_special_tokens=True)
            print(f"Chatbot: {response}")
    except EOFError:
        pass # Scenario inputs exhausted
    except Exception as e:
        print(f"Chatbot encountered an error during scenario: {e}")

    finally:
        sys.stdout = old_stdout # Restore stdout
        __builtins__.input = original_input # Restore original input
        print("--- Scenario Ended ---")
        print(redirected_output.getvalue())


# Define the scenario inputs
scenario_inputs = [
    "Hello",
    "What is Artificial Intelligence?",
    "Who created Python?",
    "Thank you",
    "exit"
]

# Run the scenario
run_scenario(tokenizer, model, scenario_inputs)

--- Scenario Ended ---

--- Running Scenario ---
Hello! I am your AI assistant. How can I help you today?
User: Hello
Chatbot: You're the best
User: What is Artificial Intelligence?
Chatbot: A bot that's smarter than you.
User: Who created Python?
Chatbot: The same guy who created Java.
User: Thank you
Chatbot: No problem. Thanks for the gold!
User: exit
Chatbot: Goodbye!



### 5. Build a Gradio Interface

Now, let's create a web-based interface for our chatbot using Gradio. This will allow for easier interaction without needing to run it in the console directly.

We'll define a `chatbot_response` function that encapsulates the model's logic for a single turn, maintaining the conversation history within the Gradio state.

In [13]:
import gradio as gr

# Global variable to store chat history for Gradio
# In a real-world app, this would be managed per user session
global_chat_history_ids = None

def chatbot_response_gradio(user_message, history):
    global global_chat_history_ids
    max_sequence_length = 512 # Same as in console version

    try:
        # Encode the new user input
        new_input_ids = tokenizer.encode(user_message + tokenizer.eos_token, return_tensors='pt')

        if global_chat_history_ids is None:
            bot_input_ids = new_input_ids
        else:
            current_history_len = global_chat_history_ids.shape[-1]
            new_input_len = new_input_ids.shape[-1]

            if current_history_len + new_input_len > max_sequence_length:
                global_chat_history_ids = global_chat_history_ids[:, -(max_sequence_length - new_input_len):]
            bot_input_ids = torch.cat([global_chat_history_ids, new_input_ids], dim=-1)

        # Generate a response
        global_chat_history_ids = model.generate(
            bot_input_ids,
            max_new_tokens=100,
            pad_token_id=tokenizer.eos_token_id,
            no_repeat_ngram_size=3,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=0.7
        )

        # Decode the response
        response = tokenizer.decode(global_chat_history_ids[:, bot_input_ids.shape[-1]:][0], skip_special_tokens=True)

        # Gradio expects history as a list of tuples (user_message, bot_response)
        history.append((user_message, response))
        return "", history

    except Exception as e:
        error_message = f"Chatbot encountered an error: {e}. Please try again."
        history.append((user_message, error_message))
        return "", history


def reset_chatbot():
    global global_chat_history_ids
    global_chat_history_ids = None
    return [], [] # Clear the Gradio chatbot history and the message box


# Check if tokenizer and model are loaded before creating the interface
if tokenizer and model:
    with gr.Blocks() as demo:
        gr.Markdown("# Hugging Face Chatbot with DialoGPT-medium")
        gr.Markdown("Ask me anything! Type 'exit' or 'quit' in the console version to stop. For this Gradio UI, use the 'Clear' button to reset the conversation.")

        chatbot = gr.Chatbot(label="Chat History", height=400)
        msg = gr.Textbox(label="Your Message", placeholder="Type your message here...")
        clear = gr.Button("Clear")

        msg.submit(chatbot_response_gradio, [msg, chatbot], [msg, chatbot])
        clear.click(reset_chatbot, [], [msg, chatbot])

    # Launch the Gradio interface
    print("\n--- Launching Gradio Interface ---")
    demo.launch(debug=True, share=True)
else:
    print("Gradio interface cannot be launched. Model or tokenizer failed to load.")

/tmp/ipykernel_5090/2767520665.py:62: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label="Chat History", height=400)
/tmp/ipykernel_5090/2767520665.py:62: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(label="Chat History", height=400)



--- Launching Gradio Interface ---
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://d7535b2dbf2f20bfa0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://d7535b2dbf2f20bfa0.gradio.live
